# Grazing-Incidence sin²ψ Strain Analysis

Compute biaxial strain (and optionally stress) from a single
grazing-incidence detector image using the `sin²ψ` method:

1. **GI polar integration** — `integrate_gi_polar` produces an
   `(q, χ)` map corrected for the grazing-incidence geometry.
2. **χ-sector extraction** — `extract_chi_sectors` slices the map
   into ψ-binned 1D profiles.
3. **Per-sector peak fit** — `fit_peak_vs_psi` fits the same peak in
   every sector, returning q_peak(ψ).
4. **Regression** — `sin2psi_regression` fits
   `d(ψ) = d₀ + slope · sin²ψ`; slope/d₀ → strain; optional `(E, ν)`
   → stress.

Or, one-call: `sin2psi_analysis(...)` does all four steps.


## 1. Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from ssrl_xrd_tools.io.image import read_image, load_mask
from ssrl_xrd_tools.io.metadata import read_txt_metadata
from ssrl_xrd_tools.integrate import load_poni, integrate_gi_polar
from ssrl_xrd_tools.integrate.calibration import poni_to_fiber_integrator
from ssrl_xrd_tools.analysis.strain import (
    extract_chi_sectors,
    fit_peak_vs_psi,
    sin2psi_regression,
    sin2psi_analysis,
)

## 2. Configuration

In [ ]:
# ───────── EDIT THESE ─────────
data_path = Path('~/data/sin2psi').expanduser()
poni_file = data_path / 'calibration.poni'
data_file = data_path / 'sample_asDep.tif'
meta_file = data_path / 'sample_asDep.txt'   # incidence angle, etc.
mask_file = data_path / 'mask.edf'

# Peak of interest: q-window around the (hkl) being analysed.
q_range = (3.0, 3.48)

# Sector configuration
chi_width  = 5.0                          # degrees per ψ bin
n_sectors  = 15                           # number of ψ bins

# Optional elastic constants for stress output (same units as the
# returned stress — use GPa here and you'll get GPa back).
E   = 70.0                                # Young's modulus
nu  = 0.34                                # Poisson's ratio
# ──────────────────────────────

## 3. Load image + build FiberIntegrator

In [ ]:
poni = load_poni(poni_file)
mask = load_mask(mask_file) if mask_file.exists() else None
data = read_image(data_file, mask=mask)
meta = read_txt_metadata(meta_file) if meta_file.exists() else {}

# The fiber integrator handles GI geometry corrections
incidence_angle = float(meta.get('th', 0.5))     # incidence angle in degrees
fi = poni_to_fiber_integrator(poni, incidence_angle=incidence_angle)
print(f'Loaded image {data.shape}, incidence = {incidence_angle}°')

## 4. GI polar map (q_total vs. χ)

In [ ]:
polar = integrate_gi_polar(
    data, fi,
    npt_rad=800,
    npt_azim=360,
    mask=mask,
    radial_range=q_range,
)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.pcolormesh(
    polar.radial, polar.azimuthal, polar.intensity.T,
    shading='auto', cmap='viridis',
)
ax.set_xlabel(f'q ({polar.unit})'); ax.set_ylabel('χ (deg)')
ax.set_title('GI polar map')
plt.colorbar(im, ax=ax, label='Intensity')
plt.tight_layout(); plt.show()

## 5. Extract χ sectors + fit the peak in each

`fit_peak_vs_psi` returns a list of `PeakFitResult` objects; each
carries `psi`, `sin2psi`, `q_center`, `d_spacing`, and their 1-σ
uncertainties.

In [ ]:
sectors = extract_chi_sectors(
    polar,
    chi_width=chi_width,
    n_sectors=n_sectors,
)
print(f'Extracted {len(sectors)} ψ sectors')

peak_fits = fit_peak_vs_psi(
    sectors,
    q_range=q_range,
    model='pseudovoigt',
    background='linear',
)
psis = np.array([pf.psi      for pf in peak_fits])
qcs  = np.array([pf.q_center for pf in peak_fits])
qerr = np.array([pf.q_center_err for pf in peak_fits])
print(f'  ψ range : {psis.min():.1f}° → {psis.max():.1f}°')
print(f'  q_peak  : {qcs.min():.4f} → {qcs.max():.4f} Å⁻¹')

## 6. sin²ψ regression → d₀, slope, optional stress

`sin2psi_regression` takes the list of `PeakFitResult` (NOT raw
arrays).  Supply `E` and `nu` to get a stress value back in the same
units as `E`.

In [ ]:
result = sin2psi_regression(peak_fits, E=E, nu=nu)
print(f'  d_0     = {result.d0:.6f} ± {result.d0_err:.6f} Å')
print(f'  slope   = {result.slope:.4e} ± {result.slope_err:.4e} Å / sin²ψ')
print(f'  R²      = {result.r_squared:.4f}')
if result.stress is not None:
    print(f'  stress  = {result.stress:.2f} ± {result.stress_err:.2f} '
          f'(units of E, here GPa)')
# Strain along the in-plane direction is conventionally (slope / d0)
print(f'  ε(sin²ψ→1) ≈ slope / d_0 = {result.slope / result.d0 * 100:.3f} %')

## 7. Diagnostic plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# d vs sin²ψ — the regression
ax = axes[0]
ax.errorbar(result.sin2psi, result.d_values,
            yerr=result.d_errors, fmt='o', label='per-sector')
fitline = np.linspace(result.sin2psi.min(), result.sin2psi.max(), 50)
ax.plot(fitline, result.d0 + result.slope * fitline, 'r-',
        label=f'fit:  d = {result.d0:.4f}{result.slope:+.2e}·sin²ψ')
ax.set_xlabel('sin²ψ'); ax.set_ylabel('d (Å)')
ax.set_title(f'sin²ψ regression  (R²={result.r_squared:.3f})')
ax.legend(fontsize=9)

# Per-sector q vs ψ
ax = axes[1]
ax.errorbar(psis, qcs, yerr=qerr, fmt='o-')
ax.set_xlabel('ψ (deg)'); ax.set_ylabel('q_center (Å⁻¹)')
ax.set_title('peak position vs ψ')

plt.tight_layout(); plt.show()

## 8. One-call shortcut

`sin2psi_analysis(result2d, q_range, ...)` chains the previous four
steps.  Note it takes the **polar map** (`IntegrationResult2D`) — not
the raw image — so the GI integration step still lives outside.

In [ ]:
one_shot = sin2psi_analysis(
    polar,
    q_range=q_range,
    chi_width=chi_width,
    n_sectors=n_sectors,
    model='pseudovoigt',
    background='linear',
    E=E, nu=nu,
)
print(f'one-call d_0    = {one_shot.d0:.6f} Å')
print(f'one-call slope  = {one_shot.slope:.4e}')
print(f'one-call stress = {one_shot.stress}'
      f' (units of E={E})')

---

### Notes

- All four steps are individually scriptable, so a non-standard
  pipeline (per-sector peak constraints, multiple peaks per sector,
  film/substrate separation) is straightforward to assemble from the
  primitives.
- For multi-peak strain analysis, run `sin2psi_analysis` per peak and
  aggregate the regression slopes.
